# NudgeIQ

## Notebook 02 – Behavioral Knowledge Base

### Objective

Transform raw customer attributes into behavioural intelligence that can be used by the AI recommendation engine.

Output:

behavioral_features.csv

| Feature                   | Business Meaning                | Source Columns               |
| ------------------------- | ------------------------------- | ---------------------------- |
| Financial Stability Score | Ability to invest               | balance, housing, loan       |
| Customer Engagement Score | Interest shown during campaigns | duration, campaign, previous |
| Marketing Responsiveness  | Response to marketing           | deposit, poutcome            |
| Contact Preference        | Preferred communication method  | contact                      |
| Life Stage                | Customer maturity               | age                          |
| Financial Commitment      | Existing liabilities            | housing, loan                |
| Investor Readiness Score  | Overall readiness to invest     | Combination of above         |
| Investor Segment          | Final customer category         | Investor Readiness Score     |


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: d:\NudgeIQ


In [3]:
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR
from src.utils import load_dataset, print_section

config.py loaded


In [4]:
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR
from src.utils import load_dataset, print_section

In [5]:
print_section("Loading Behavioural Dataset")

df = load_dataset(
    PROCESSED_DIR / "behavioral_data.csv",
    separator=","
)

print(df.shape)

df.head()


Loading Behavioural Dataset
(11162, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [6]:
print_section("Creating Life Stage")

def life_stage(age):

    if age < 30:
        return "Young"

    elif age < 45:
        return "Early Career"

    elif age < 60:
        return "Mid Career"

    else:
        return "Senior"


df["Life_Stage"] = df["age"].apply(life_stage)

df["Life_Stage"].value_counts()


Creating Life Stage


Life_Stage
Early Career    5725
Mid Career      3106
Young           1551
Senior           780
Name: count, dtype: int64

In [7]:
print_section("Financial Commitment")

conditions = [
    (df["housing"]=="no") & (df["loan"]=="no"),
    (df["housing"]=="yes") & (df["loan"]=="no"),
    (df["housing"]=="yes") & (df["loan"]=="yes"),
]

choices = [
    "Low",
    "Medium",
    "High"
]

df["Financial_Commitment"] = np.select(
    conditions,
    choices,
    default="Medium"
)

df["Financial_Commitment"].value_counts()


Financial Commitment


Financial_Commitment
Low       5256
Medium    5071
High       835
Name: count, dtype: int64

In [8]:
from sklearn.preprocessing import RobustScaler

print_section("Financial Stability Score")

# Keep original balance
df["Balance_Original"] = df["balance"]

# Robust scaling (less affected by outliers)
scaler = RobustScaler()

df["Balance_Scaled"] = scaler.fit_transform(df[["balance"]])

df[["balance", "Balance_Scaled"]].head()


Financial Stability Score


,balance,Balance_Scaled
0,2343,1.130517
1,45,-0.318411
2,1270,0.453972
3,2476,1.214376
4,184,-0.230769


In [9]:
print_section("Financial Stability Label")

quartiles = df["balance"].quantile([0.25, 0.50, 0.75])

Q1 = quartiles[0.25]
Q2 = quartiles[0.50]
Q3 = quartiles[0.75]


def financial_label(balance):

    if balance <= Q1:
        return "Low"

    elif balance <= Q2:
        return "Moderate"

    elif balance <= Q3:
        return "High"

    else:
        return "Very High"


df["Financial_Stability"] = df["balance"].apply(financial_label)

df["Financial_Stability"].value_counts()



Financial Stability Label


Financial_Stability
Low          2792
Moderate     2791
Very High    2790
High         2789
Name: count, dtype: int64

In [10]:
print_section("Customer Engagement Score")

duration_scaled = (
    df["duration"] - df["duration"].min()
) / (
    df["duration"].max() - df["duration"].min()
)

previous_scaled = (
    df["previous"] - df["previous"].min()
) / (
    df["previous"].max() - df["previous"].min()
)

campaign_scaled = (
    df["campaign"] - df["campaign"].min()
) / (
    df["campaign"].max() - df["campaign"].min()
)

df["Engagement_Score"] = (
    0.5 * duration_scaled +
    0.3 * previous_scaled +
    0.2 * (1 - campaign_scaled)
)

df["Engagement_Score"] *= 100

df["Engagement_Score"] = df["Engagement_Score"].round(1)

df[[
    "duration",
    "campaign",
    "previous",
    "Engagement_Score"
]].head()


Customer Engagement Score


,duration,campaign,previous,Engagement_Score
0,1042,1,0,33.4
1,1467,1,0,38.9
2,1389,1,0,37.9
3,579,1,0,27.4
4,673,2,0,28.3


In [11]:
quartiles = df["Engagement_Score"].quantile([0.25,0.50,0.75])

Q1,Q2,Q3 = quartiles


def engagement_label(score):

    if score <= Q1:
        return "Low"

    elif score <= Q2:
        return "Medium"

    elif score <= Q3:
        return "High"

    else:
        return "Very High"


df["Engagement_Level"] = df["Engagement_Score"].apply(
    engagement_label
)

df["Engagement_Level"].value_counts()

Engagement_Level
Medium       2850
Low          2844
Very High    2753
High         2715
Name: count, dtype: int64

# Reusable Behavioural Score Framework

To ensure consistency across behavioural feature engineering, NudgeIQ implements a reusable behavioural scoring framework.

The framework:

- scales behavioural variables
- optionally inverts negatively associated variables
- applies configurable weights
- generates a continuous behavioural score
- creates business-friendly labels

This framework will be reused in subsequent notebooks for Investor Twin Generation and Recommendation Modelling.

In [12]:
from sklearn.preprocessing import RobustScaler
import numpy as np
import pandas as pd

In [13]:
def create_behavioral_score(
    data,
    columns,
    weights=None,
    invert=None,
    scaler=None
):
    """
    Generic Behavioural Score Generator

    Parameters
    ----------
    data : DataFrame

    columns : list
        Behavioural variables

    weights : list
        Variable importance weights

    invert : list
        Variables where lower values are better

    scaler : sklearn scaler
        Defaults to RobustScaler()

    Returns
    -------
    Continuous behavioural score (0-100)
    """

    if scaler is None:
        scaler = RobustScaler()

    scaled = pd.DataFrame(
        scaler.fit_transform(data[columns]),
        columns=columns,
        index=data.index
    )

    if invert is not None:

        for col in invert:
            scaled[col] *= -1

    if weights is None:

        weights = np.ones(len(columns))

    weights = np.array(weights)

    weights = weights / weights.sum()

    score = scaled.dot(weights)

    return score

In [14]:
engagement_columns = [
    "previous",
    "campaign"
]

engagement_weights = [
    0.5,
    0.5
]

df["Engagement_Score_Framework"] = create_behavioral_score(
    data=df,
    columns=engagement_columns,
    weights=engagement_weights,
    invert=["campaign"]
)